In [ ]:
from pathlib import Path

from app.session import ReadingSession
from app.environment import (
    DATA_DIR,
    initialize_environment,
)
from app.config import LLM_PROVIDER, OPENAI_MODEL
from app.llm import create_llm


In [2]:
BOOK_NAME = "Harry Potter and the Sorcerer's Stone.epub"
LAST_READ_SENTENCE = "I bet he asked Dumbledore to keep it safe for him"
QUERY = "who is Harry"


initialize_environment()
llm = create_llm(model=OPENAI_MODEL, provider=LLM_PROVIDER)

session = ReadingSession(llm=llm)

session.load_book(DATA_DIR / BOOK_NAME)

session.set_position(LAST_READ_SENTENCE)



2026-03-29 16:38:10,114 - INFO - Loading all indices.
2026-03-29 16:38:10,930 - INFO - Loading all indices.


ResolvedPosition(snippet='I bet he asked Dumbledore to keep it safe for him', normalized_snippet='i bet he asked dumbledore to keep it safe for him', start_char=312281, end_char=312330)

In [ ]:
print(session.ask(QUERY))

In [5]:
position = session.position

print("position:", position)
print("position.reader_boundary_key:", position.reader_boundary_key)

nodes = session.nodes or []

print("total nodes:", len(nodes))
print("first 5 node end_char_idx values:")
for node in nodes[:5]:
    print(getattr(node, "end_char_idx", None))

position: ResolvedPosition(snippet='I bet he asked Dumbledore to keep it safe for him', normalized_snippet='i bet he asked dumbledore to keep it safe for him', start_char=312281, end_char=312330)
position.end_char: 312330
total nodes: 145
first 5 node end_char_idx values:
3768
6865
9820
12768
15812


In [ ]:
# Quick manual spoiler-free checks at different positions

TEST_CASES = [
    {
        "last_read_snippet": "I bet he asked Dumbledore to keep it safe for him",
        "question": "Who is Harry?",
        "description": "Early-story factual question that should be answerable without spoilers.",
    },
    {
        "last_read_snippet": "I bet he asked Dumbledore to keep it safe for him",
        "question": "Why did Snape die?",
        "description": "Question about a much later plot event; should trigger spoiler refusal.",
    },
]

for case in TEST_CASES:
    print("\n=== Test case ===")
    print("Description:", case["description"])
    print("Last read snippet:", case["last_read_snippet"])
    print("Question:", case["question"])

    # Reset position for each test case
    position = session.set_position(case["last_read_snippet"])
    print("Resolved position reader_boundary_key:", position.reader_boundary_key)

    answer = session.ask(case["question"])
    print("Answer:\n", answer)